In [ ]:
"""Setup environment variables for nnFormer."""
import os
import sys

# Set nnFormer environment variables
NNFORMER_ROOT = os.environ["NNFORMER_ROOT"]
os.environ['nnFormer_raw_data_base'] = os.path.join(NNFORMER_ROOT, "DATASET", "nnFormer_raw")
os.environ['nnFormer_preprocessed'] = os.path.join(NNFORMER_ROOT, "DATASET", "nnFormer_preprocessed")
os.environ['RESULTS_FOLDER'] = os.path.join(NNFORMER_ROOT, "DATASET", "nnFormer_trained_models")

# Add nnFormer to path
sys.path.insert(0, NNFORMER_ROOT)

print(f"nnFormer_raw_data_base: {os.environ['nnFormer_raw_data_base']}")
print(f"nnFormer_preprocessed: {os.environ['nnFormer_preprocessed']}")
print(f"RESULTS_FOLDER: {os.environ['RESULTS_FOLDER']}")

In [ ]:
"""Import required libraries."""
from datetime import datetime
from pathlib import Path
import glob
import json
import math
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
from medpy.metric import binary
from tqdm import tqdm

# print(f"PyTorch version: {torch.__version__}")
# print(f"CUDA available: {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Configuration

Set paths for ACDC data, model, and output directories.

In [ ]:
# Configuration
NNFORMER_ROOT = Path(os.environ["NNFORMER_ROOT"])

# Input data paths
IMAGES_TS_DIR = NNFORMER_ROOT / "DATASET" / "nnFormer_raw" / "nnFormer_raw_data" / "Task01_ACDC" / "imagesTs"
LABELS_TS_DIR = NNFORMER_ROOT / "DATASET" / "nnFormer_raw" / "nnFormer_raw_data" / "Task01_ACDC" / "labelsTs"

# Model path
MODEL_DIR = NNFORMER_ROOT / "DATASET" / "nnFormer_trained_models" / "nnFormer" / "3d_fullres" / "Task001_ACDC" / "nnFormerTrainerV2_nnformer_acdc__nnFormerPlansv2.1"

# Output directory
RESULTS_DIR = NNFORMER_ROOT / "results"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"
VISUALIZATIONS_DIR = RESULTS_DIR / "visualizations"

# CASC-style debug output directories
DEBUG_DIR = RESULTS_DIR / "debug"
DEBUG_TEXT_DIR = DEBUG_DIR / "text"
DEBUG_FIGURES_DIR = DEBUG_DIR / "figures"
DEBUG_RUN_NAME = "notebook_nnformer_acdc_testing"

# Create output directories
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZATIONS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_TEXT_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input images: {IMAGES_TS_DIR}")
print(f"Ground truth labels: {LABELS_TS_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"Results will be saved to: {RESULTS_DIR}")
print(f"Debug outputs will be saved to: {DEBUG_DIR}")

## Helper Functions

Define utility functions for loading images and computing metrics.

In [ ]:
def read_nii(path):
    """Read NIfTI image and return array with spacing."""
    itk_img = sitk.ReadImage(str(path))
    spacing = np.array(itk_img.GetSpacing())
    return sitk.GetArrayFromImage(itk_img), spacing, itk_img


def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    """Calculate Dice score for binary masks.
    
    Args:
        pred: predicted binary segmentation
        gt: ground truth binary segmentation
        
    Returns:
        Dice score value
    """
    if (pred.sum() + gt.sum()) == 0:
        return 1.0
    return 2.0 * np.logical_and(pred, gt).sum() / (pred.sum() + gt.sum())


def hd95_score(pred: np.ndarray, gt: np.ndarray, voxelspacing=None) -> float:
    """Calculate 95th percentile Hausdorff distance.
    
    Args:
        pred: predicted binary segmentation
        gt: ground truth binary segmentation
        voxelspacing: voxel spacing for physical distance calculation
        
    Returns:
        HD95 value in mm (or voxels if no spacing provided)
    """
    if pred.sum() > 0 and gt.sum() > 0:
        return binary.hd95(pred, gt, voxelspacing=voxelspacing)
    elif pred.sum() == 0 and gt.sum() == 0:
        return 0.0
    else:
        # One is empty, the other is not - return a large value
        return np.inf


def process_label(label):
    """Extract binary masks for each cardiac structure.
    
    ACDC labels:
        - 1: Right Ventricle (RV)
        - 2: Myocardium (MYO)
        - 3: Left Ventricle (LV)
    """
    rv = (label == 1).astype(np.uint8)
    myo = (label == 2).astype(np.uint8)
    lv = (label == 3).astype(np.uint8)
    return rv, myo, lv


def get_patient_frame_info(filename):
    """Extract patient ID and frame number from filename.
    
    Args:
        filename: e.g., 'patient101_frame01.nii.gz' or 'patient101_frame01_gt.nii.gz'
        
    Returns:
        tuple: (patient_id, frame_number, frame_type)
        frame_type is 'ED' for frame01, 'ES' for other frames
    """
    match = re.match(r'(patient\d+)_frame(\d+)', filename)
    if match:
        patient_id = match.group(1)
        frame_num = int(match.group(2))
        # Frame 01 is typically ED (End-Diastole), other frames are ES (End-Systole)
        frame_type = 'ED' if frame_num == 1 else 'ES'
        return patient_id, frame_num, frame_type
    return None, None, None


print("Helper functions loaded successfully.")

In [ ]:
# Debug tracking helpers (CASC-style report artifacts)
try:
    import resource  # Linux/Unix only
except ImportError:
    resource = None


def _safe_float(value):
    if value is None:
        return float("nan")
    try:
        value = float(value)
    except (TypeError, ValueError):
        return float("nan")
    return value if math.isfinite(value) else float("nan")


def _nan_to_na(value, precision=4):
    value = _safe_float(value)
    if math.isnan(value):
        return "NA"
    return f"{value:.{precision}f}"


def _get_peak_rss_gb():
    if resource is None:
        return float("nan")
    usage = resource.getrusage(resource.RUSAGE_SELF)
    # On Linux ru_maxrss is KiB.
    return _safe_float(usage.ru_maxrss) / (1024.0 * 1024.0)


class NotebookDebugTracker:
    def __init__(self):
        self.workflow_start_iso = datetime.now().astimezone().isoformat()
        self._workflow_t0 = time.perf_counter()
        self.records = []

    def start_task(self, process_name, gpu_task=False):
        return {
            "process": process_name,
            "gpu_task": bool(gpu_task),
            "start_perf": time.perf_counter(),
            "start_peak_rss_gb": _get_peak_rss_gb(),
        }

    def end_task(self, task_handle, success=True):
        end_perf = time.perf_counter()
        runtime_s = _safe_float(end_perf - task_handle["start_perf"])
        peak_rss_gb = max(_safe_float(task_handle["start_peak_rss_gb"]), _safe_float(_get_peak_rss_gb()))
        self.records.append(
            {
                "process": task_handle["process"],
                "success": bool(success),
                "runtime_s": runtime_s,
                "cpu_percent": float("nan"),
                "peak_rss_gb": peak_rss_gb,
                "gpu_task": bool(task_handle["gpu_task"]),
            }
        )

    def workflow_duration_seconds(self):
        return _safe_float(time.perf_counter() - self._workflow_t0)


DEBUG_TRACKER = NotebookDebugTracker()

## Step 1: Run nnFormer Inference

Run segmentation inference using the nnFormer model on all test images.

In [ ]:
# List all test images
image_files = sorted(IMAGES_TS_DIR.glob("*.nii.gz"))
print(f"Found {len(image_files)} test images")

# Get unique patients
patients = set()
for f in image_files:
    patient_id, _, _ = get_patient_frame_info(f.name)
    if patient_id:
        patients.add(patient_id)
print(f"Found {len(patients)} unique patients")

# Display first few files
print("\nSample image files:")
for f in image_files[:4]:
    print(f"  {f.name}")

In [ ]:
import pickle
import shutil

model_pkl_path = MODEL_DIR / "fold_0" / "model_best.model.pkl"
plans_path = MODEL_DIR / "plans.pkl"

# Load plans from model checkpoint
with open(model_pkl_path, 'rb') as f:
    model_pkl = pickle.load(f)

plans = model_pkl['plans']
print(f"Extracted plans from: {model_pkl_path}")
print(f"  num_modalities: {plans['num_modalities']}")
print(f"  num_classes: {plans['num_classes']}")
print(f"  plans_per_stage keys: {list(plans.get('plans_per_stage', {}).keys()) if isinstance(plans.get('plans_per_stage'), dict) else 'N/A'}")

In [ ]:
import shutil
import importlib

import nnformer.paths
importlib.reload(nnformer.paths)

import nnformer.training.model_restore
importlib.reload(nnformer.training.model_restore)

import nnformer.inference.predict
importlib.reload(nnformer.inference.predict)

# Verify paths are set correctly
print("Verifying nnFormer paths after reload:")
print(f"  preprocessing_output_dir: {nnformer.paths.preprocessing_output_dir}")
print(f"  default_plans_identifier: {nnformer.paths.default_plans_identifier}")
print(f"  network_training_output_dir: {nnformer.paths.network_training_output_dir}")

# Verify model_restore has the updated paths
from nnformer.training.model_restore import preprocessing_output_dir as mr_prep_dir
from nnformer.training.model_restore import default_plans_identifier as mr_plans_id
print(f"\nmodel_restore module paths:")
print(f"  preprocessing_output_dir: {mr_prep_dir}")
print(f"  default_plans_identifier: {mr_plans_id}")

from nnformer.inference.predict import predict_from_folder

# Need to create symlinks: patient101_frame01_0000.nii.gz -> patient101_frame01.nii.gz

# Create a temporary input directory with properly named symlinks
TEMP_INPUT_DIR = RESULTS_DIR / "temp_input"
TEMP_INPUT_DIR.mkdir(parents=True, exist_ok=True)

symlink_task = DEBUG_TRACKER.start_task("CREATE_TEMP_INPUT_SYMLINKS", gpu_task=False)
try:
    # Create symlinks with _0000 suffix
    print("\nCreating symlinks with nnFormer naming convention...")
    for img_file in sorted(IMAGES_TS_DIR.glob("*.nii.gz")):
        # patient101_frame01.nii.gz -> patient101_frame01_0000.nii.gz
        new_name = img_file.stem.replace('.nii', '') + "_0000.nii.gz"
        symlink_path = TEMP_INPUT_DIR / new_name
        
        # Remove existing symlink if present
        if symlink_path.exists() or symlink_path.is_symlink():
            symlink_path.unlink()
        
        # Create symlink
        symlink_path.symlink_to(img_file)
    DEBUG_TRACKER.end_task(symlink_task, success=True)
except Exception:
    DEBUG_TRACKER.end_task(symlink_task, success=False)
    raise

print(f"Created {len(list(TEMP_INPUT_DIR.glob('*_0000.nii.gz')))} symlinks in {TEMP_INPUT_DIR}")

# Output directory for nnFormer predictions
NNFORMER_OUTPUT_DIR = RESULTS_DIR / "nnformer_predictions"
NNFORMER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Run inference
print("\nRunning nnFormer inference...")
infer_task = DEBUG_TRACKER.start_task("NNFORMER_INFERENCE", gpu_task=torch.cuda.is_available())
try:
    predict_from_folder(
        model=str(MODEL_DIR),
        input_folder=str(TEMP_INPUT_DIR),
        output_folder=str(NNFORMER_OUTPUT_DIR),
        folds=[0],
        save_npz=False,
        num_threads_preprocessing=4,
        num_threads_nifti_save=2,
        lowres_segmentations=None,
        part_id=0,
        num_parts=1,
        tta=True,
        mixed_precision=True,
        overwrite_existing=True,
        mode='normal',
        step_size=0.5,
        checkpoint_name="model_best"
    )
    DEBUG_TRACKER.end_task(infer_task, success=True)
except Exception:
    DEBUG_TRACKER.end_task(infer_task, success=False)
    raise

print(f"\n✓ Inference complete! Predictions saved to: {NNFORMER_OUTPUT_DIR}")

## Step 2: Compute Evaluation Metrics

Calculate Dice scores and HD95 for each structure (RV, MYO, LV) and organize by patient with ED/ES breakdown.

In [ ]:
"""Compute metrics for all predictions."""

# Get all prediction files (from nnFormer's output directory)
pred_files = sorted(NNFORMER_OUTPUT_DIR.glob("*.nii.gz"))
print(f"Found {len(pred_files)} prediction files")

# Initialize results list
all_results = []

for pred_path in tqdm(pred_files, desc="Computing metrics"):
    metric_task = DEBUG_TRACKER.start_task(f"COMPUTE_METRICS ({pred_path.name})", gpu_task=False)
    
    pred_name = pred_path.name
    gt_filename = pred_name.replace('.nii.gz', '_gt.nii.gz')
    gt_path = LABELS_TS_DIR / gt_filename
    
    if not gt_path.exists():
        DEBUG_TRACKER.end_task(metric_task, success=False)
        print(f"Warning: Ground truth not found for {pred_path.name} (looking for {gt_filename})")
        continue
    
    try:
        # Extract patient and frame info from prediction filename
        patient_id, frame_num, frame_type = get_patient_frame_info(pred_name)
        
        # Load images
        pred_array, spacing, _ = read_nii(pred_path)
        gt_array, _, _ = read_nii(gt_path)
        
        # Get voxel spacing for HD95 (in mm)
        # spacing is (x, y, z) in SimpleITK
        voxelspacing = spacing[::-1]  # Convert to (z, y, x) for numpy array
        
        # Extract binary masks
        pred_rv, pred_myo, pred_lv = process_label(pred_array)
        gt_rv, gt_myo, gt_lv = process_label(gt_array)
        
        # Calculate Dice scores
        dice_rv = dice_score(pred_rv, gt_rv)
        dice_myo = dice_score(pred_myo, gt_myo)
        dice_lv = dice_score(pred_lv, gt_lv)
        
        # Calculate HD95
        hd95_rv = hd95_score(pred_rv, gt_rv, voxelspacing=voxelspacing)
        hd95_myo = hd95_score(pred_myo, gt_myo, voxelspacing=voxelspacing)
        hd95_lv = hd95_score(pred_lv, gt_lv, voxelspacing=voxelspacing)
        
        # Store results
        result = {
            'patient_id': patient_id,
            'frame_num': frame_num,
            'frame_type': frame_type,
            'filename': pred_name,
            'dice_rv': dice_rv,
            'dice_myo': dice_myo,
            'dice_lv': dice_lv,
            'dice_mean': np.mean([dice_rv, dice_myo, dice_lv]),
            'hd95_rv': hd95_rv,
            'hd95_myo': hd95_myo,
            'hd95_lv': hd95_lv,
            'hd95_mean': np.mean([hd95_rv, hd95_myo, hd95_lv]) if all(np.isfinite([hd95_rv, hd95_myo, hd95_lv])) else np.inf,
        }
        all_results.append(result)
        DEBUG_TRACKER.end_task(metric_task, success=True)
    except Exception:
        DEBUG_TRACKER.end_task(metric_task, success=False)
        raise

# Create DataFrame
results_df = pd.DataFrame(all_results)
print(f"\n✓ Computed metrics for {len(results_df)} images")

In [ ]:
"""Organize results by patient with ED/ES breakdown."""

# Create patient-level summary
patient_results = []

for patient_id in sorted(results_df['patient_id'].unique()):
    patient_data = results_df[results_df['patient_id'] == patient_id]
    
    # Get ED and ES data
    ed_data = patient_data[patient_data['frame_type'] == 'ED']
    es_data = patient_data[patient_data['frame_type'] == 'ES']
    
    patient_result = {'patient_id': patient_id}
    
    # ED metrics
    if len(ed_data) > 0:
        ed_row = ed_data.iloc[0]
        patient_result['ed_dice_rv'] = ed_row['dice_rv']
        patient_result['ed_dice_myo'] = ed_row['dice_myo']
        patient_result['ed_dice_lv'] = ed_row['dice_lv']
        patient_result['ed_dice_mean'] = ed_row['dice_mean']
        patient_result['ed_hd95_rv'] = ed_row['hd95_rv']
        patient_result['ed_hd95_myo'] = ed_row['hd95_myo']
        patient_result['ed_hd95_lv'] = ed_row['hd95_lv']
        patient_result['ed_hd95_mean'] = ed_row['hd95_mean']
    
    # ES metrics
    if len(es_data) > 0:
        es_row = es_data.iloc[0]
        patient_result['es_dice_rv'] = es_row['dice_rv']
        patient_result['es_dice_myo'] = es_row['dice_myo']
        patient_result['es_dice_lv'] = es_row['dice_lv']
        patient_result['es_dice_mean'] = es_row['dice_mean']
        patient_result['es_hd95_rv'] = es_row['hd95_rv']
        patient_result['es_hd95_myo'] = es_row['hd95_myo']
        patient_result['es_hd95_lv'] = es_row['hd95_lv']
        patient_result['es_hd95_mean'] = es_row['hd95_mean']
    
    # Overall patient metrics
    patient_result['overall_dice_mean'] = patient_data['dice_mean'].mean()
    finite_hd95 = patient_data['hd95_mean'][np.isfinite(patient_data['hd95_mean'])]
    patient_result['overall_hd95_mean'] = finite_hd95.mean() if len(finite_hd95) > 0 else np.inf
    
    patient_results.append(patient_result)

patient_df = pd.DataFrame(patient_results)
print(f"Created patient-level summary for {len(patient_df)} patients")

## Step 3: Analyze and Display Results

Display summary statistics and save results to CSV files.

In [ ]:
"""Display summary statistics."""

print("=" * 80)
print("nnFORMER ACDC SEGMENTATION RESULTS - DICE SCORES")
print("=" * 80)

# ED Statistics
print("\n--- End-Diastole (ED) Dice Scores ---")
ed_cols = ['ed_dice_rv', 'ed_dice_myo', 'ed_dice_lv', 'ed_dice_mean']
for col in ed_cols:
    if col in patient_df.columns:
        mean_val = patient_df[col].mean()
        std_val = patient_df[col].std()
        label = col.replace('ed_dice_', '').upper()
        print(f"  {label}: {mean_val:.4f} ± {std_val:.4f}")

# ES Statistics
print("\n--- End-Systole (ES) Dice Scores ---")
es_cols = ['es_dice_rv', 'es_dice_myo', 'es_dice_lv', 'es_dice_mean']
for col in es_cols:
    if col in patient_df.columns:
        mean_val = patient_df[col].mean()
        std_val = patient_df[col].std()
        label = col.replace('es_dice_', '').upper()
        print(f"  {label}: {mean_val:.4f} ± {std_val:.4f}")

# Overall
print("\n--- Overall Dice Score ---")
print(f"  MEAN: {patient_df['overall_dice_mean'].mean():.4f} ± {patient_df['overall_dice_mean'].std():.4f}")

In [ ]:
"""Display HD95 statistics."""

print("=" * 80)
print("nnFORMER ACDC SEGMENTATION RESULTS - HD95 (mm)")
print("=" * 80)

# Filter out infinite values for statistics
def safe_stats(series):
    finite_vals = series[np.isfinite(series)]
    if len(finite_vals) > 0:
        return finite_vals.mean(), finite_vals.std()
    return np.nan, np.nan

# ED Statistics
print("\n--- End-Diastole (ED) HD95 ---")
ed_hd_cols = ['ed_hd95_rv', 'ed_hd95_myo', 'ed_hd95_lv', 'ed_hd95_mean']
for col in ed_hd_cols:
    if col in patient_df.columns:
        mean_val, std_val = safe_stats(patient_df[col])
        label = col.replace('ed_hd95_', '').upper()
        print(f"  {label}: {mean_val:.2f} ± {std_val:.2f} mm")

# ES Statistics
print("\n--- End-Systole (ES) HD95 ---")
es_hd_cols = ['es_hd95_rv', 'es_hd95_myo', 'es_hd95_lv', 'es_hd95_mean']
for col in es_hd_cols:
    if col in patient_df.columns:
        mean_val, std_val = safe_stats(patient_df[col])
        label = col.replace('es_hd95_', '').upper()
        print(f"  {label}: {mean_val:.2f} ± {std_val:.2f} mm")

# Overall
print("\n--- Overall HD95 ---")
mean_val, std_val = safe_stats(patient_df['overall_hd95_mean'])
print(f"  MEAN: {mean_val:.2f} ± {std_val:.2f} mm")

In [ ]:
"""Create summary table."""

# Create a summary DataFrame for easy comparison
summary_data = {
    'Metric': ['Dice RV', 'Dice MYO', 'Dice LV', 'Dice Mean', 'HD95 RV', 'HD95 MYO', 'HD95 LV', 'HD95 Mean'],
    'ED Mean': [
        patient_df['ed_dice_rv'].mean(),
        patient_df['ed_dice_myo'].mean(),
        patient_df['ed_dice_lv'].mean(),
        patient_df['ed_dice_mean'].mean(),
        safe_stats(patient_df['ed_hd95_rv'])[0],
        safe_stats(patient_df['ed_hd95_myo'])[0],
        safe_stats(patient_df['ed_hd95_lv'])[0],
        safe_stats(patient_df['ed_hd95_mean'])[0],
    ],
    'ED Std': [
        patient_df['ed_dice_rv'].std(),
        patient_df['ed_dice_myo'].std(),
        patient_df['ed_dice_lv'].std(),
        patient_df['ed_dice_mean'].std(),
        safe_stats(patient_df['ed_hd95_rv'])[1],
        safe_stats(patient_df['ed_hd95_myo'])[1],
        safe_stats(patient_df['ed_hd95_lv'])[1],
        safe_stats(patient_df['ed_hd95_mean'])[1],
    ],
    'ES Mean': [
        patient_df['es_dice_rv'].mean(),
        patient_df['es_dice_myo'].mean(),
        patient_df['es_dice_lv'].mean(),
        patient_df['es_dice_mean'].mean(),
        safe_stats(patient_df['es_hd95_rv'])[0],
        safe_stats(patient_df['es_hd95_myo'])[0],
        safe_stats(patient_df['es_hd95_lv'])[0],
        safe_stats(patient_df['es_hd95_mean'])[0],
    ],
    'ES Std': [
        patient_df['es_dice_rv'].std(),
        patient_df['es_dice_myo'].std(),
        patient_df['es_dice_lv'].std(),
        patient_df['es_dice_mean'].std(),
        safe_stats(patient_df['es_hd95_rv'])[1],
        safe_stats(patient_df['es_hd95_myo'])[1],
        safe_stats(patient_df['es_hd95_lv'])[1],
        safe_stats(patient_df['es_hd95_mean'])[1],
    ],
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)
print(summary_df.to_string(index=False))

In [ ]:
"""Save results to CSV files."""

# Save detailed results (per image)
detailed_csv_path = RESULTS_DIR / "metrics_detailed.csv"
results_df.to_csv(detailed_csv_path, index=False)
print(f"✓ Detailed results saved to: {detailed_csv_path}")

# Save patient-level results
patient_csv_path = RESULTS_DIR / "metrics_by_patient.csv"
patient_df.to_csv(patient_csv_path, index=False)
print(f"✓ Patient-level results saved to: {patient_csv_path}")

# Save summary statistics
summary_csv_path = RESULTS_DIR / "metrics_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"✓ Summary statistics saved to: {summary_csv_path}")

## Step 4: Visualize Predictions vs Ground Truth

Compare predicted segmentations with ground truth for qualitative assessment.

In [ ]:
"""Define visualization function."""

def visualize_comparison(pred_path, gt_path, image_path, patient_id, frame_type, save_path=None):
    """Visualize prediction vs ground truth comparison.
    
    Args:
        pred_path: path to prediction NIfTI
        gt_path: path to ground truth NIfTI
        image_path: path to original image NIfTI
        patient_id: patient identifier
        frame_type: 'ED' or 'ES'
        save_path: optional path to save the figure
    """
    # Load images
    pred_array, spacing, _ = read_nii(pred_path)
    gt_array, _, _ = read_nii(gt_path)
    img_array, _, _ = read_nii(image_path)
    
    # Select middle slice for visualization
    mid_slice = pred_array.shape[0] // 2
    
    pred_slice = pred_array[mid_slice]
    gt_slice = gt_array[mid_slice]
    img_slice = img_array[mid_slice]
    
    # Calculate metrics for display
    pred_rv, pred_myo, pred_lv = process_label(pred_array)
    gt_rv, gt_myo, gt_lv = process_label(gt_array)
    
    dsc_rv = dice_score(pred_rv, gt_rv)
    dsc_myo = dice_score(pred_myo, gt_myo)
    dsc_lv = dice_score(pred_lv, gt_lv)
    dsc_mean = np.mean([dsc_rv, dsc_myo, dsc_lv])
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Colormap for segmentation (0=BG, 1=RV, 2=MYO, 3=LV)
    cmap = plt.cm.colors.ListedColormap(['black', 'red', 'green', 'blue'])
    
    # Top row: Original, Prediction, Ground Truth
    axes[0, 0].imshow(img_slice, cmap='gray')
    axes[0, 0].set_title(f'Original - {patient_id} ({frame_type}, Slice {mid_slice})', fontsize=12)
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(pred_slice, cmap=cmap, vmin=0, vmax=3)
    axes[0, 1].set_title('Prediction (nnFormer)', fontsize=12)
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(gt_slice, cmap=cmap, vmin=0, vmax=3)
    axes[0, 2].set_title('Ground Truth', fontsize=12)
    axes[0, 2].axis('off')
    
    # Bottom row: Overlays and Difference
    axes[1, 0].imshow(img_slice, cmap='gray')
    pred_masked = np.ma.masked_where(pred_slice == 0, pred_slice)
    axes[1, 0].imshow(pred_masked, alpha=0.6, cmap=cmap, vmin=0, vmax=3)
    axes[1, 0].set_title('Prediction Overlay', fontsize=12)
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(img_slice, cmap='gray')
    gt_masked = np.ma.masked_where(gt_slice == 0, gt_slice)
    axes[1, 1].imshow(gt_masked, alpha=0.6, cmap=cmap, vmin=0, vmax=3)
    axes[1, 1].set_title('Ground Truth Overlay', fontsize=12)
    axes[1, 1].axis('off')
    
    # Difference map
    difference = np.abs(pred_slice.astype(int) - gt_slice.astype(int))
    axes[1, 2].imshow(difference, cmap='hot')
    axes[1, 2].set_title(f'Difference Map\nMean DSC: {dsc_mean:.3f}', fontsize=12)
    axes[1, 2].axis('off')
    
    # Add legend
    legend_elements = [
        plt.Rectangle((0,0),1,1, facecolor='red', label=f'RV (DSC={dsc_rv:.3f})'),
        plt.Rectangle((0,0),1,1, facecolor='green', label=f'MYO (DSC={dsc_myo:.3f})'),
        plt.Rectangle((0,0),1,1, facecolor='blue', label=f'LV (DSC={dsc_lv:.3f})'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11)
    
    plt.suptitle(f'{patient_id} - {frame_type} Frame - DSC: RV={dsc_rv:.3f} | MYO={dsc_myo:.3f} | LV={dsc_lv:.3f}', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved: {save_path}")
    
    plt.show()


print("Visualization function loaded.")

In [ ]:
"""Visualize predictions for sample patients."""

# Select a few patients to visualize
sample_patients = sorted(patient_df['patient_id'].unique())[:3]
print(f"Visualizing predictions for: {sample_patients}\n")

for patient_id in sample_patients:
    print(f"\n{'='*80}")
    print(f"Patient: {patient_id}")
    print(f"{'='*80}")
    
    # Get patient data
    patient_data = results_df[results_df['patient_id'] == patient_id]
    
    for _, row in patient_data.iterrows():
        frame_type = row['frame_type']
        filename = row['filename']
        
        # File paths - use NNFORMER_OUTPUT_DIR for predictions
        pred_path = NNFORMER_OUTPUT_DIR / filename
        gt_filename = filename.replace('.nii.gz', '_gt.nii.gz')
        gt_path = LABELS_TS_DIR / gt_filename
        img_path = IMAGES_TS_DIR / filename
        
        if pred_path.exists() and gt_path.exists() and img_path.exists():
            # Save visualization
            save_path = VISUALIZATIONS_DIR / f"{patient_id}_{frame_type}_comparison.png"
            visualize_comparison(pred_path, gt_path, img_path, patient_id, frame_type, save_path)

In [ ]:
"""Create box plots for Dice scores and HD95."""

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Dice scores - ED
dice_ed_data = [patient_df['ed_dice_rv'], patient_df['ed_dice_myo'], patient_df['ed_dice_lv']]
bp1 = axes[0, 0].boxplot(dice_ed_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
colors = ['red', 'green', 'blue']
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[0, 0].set_title('ED Dice Scores', fontsize=12)
axes[0, 0].set_ylabel('Dice Score')
axes[0, 0].set_ylim([0, 1])
axes[0, 0].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)

# Dice scores - ES
dice_es_data = [patient_df['es_dice_rv'], patient_df['es_dice_myo'], patient_df['es_dice_lv']]
bp2 = axes[0, 1].boxplot(dice_es_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[0, 1].set_title('ES Dice Scores', fontsize=12)
axes[0, 1].set_ylabel('Dice Score')
axes[0, 1].set_ylim([0, 1])
axes[0, 1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)

# HD95 - ED (filter out infinite values)
hd95_ed_data = [
    patient_df['ed_hd95_rv'][np.isfinite(patient_df['ed_hd95_rv'])],
    patient_df['ed_hd95_myo'][np.isfinite(patient_df['ed_hd95_myo'])],
    patient_df['ed_hd95_lv'][np.isfinite(patient_df['ed_hd95_lv'])]
]
bp3 = axes[1, 0].boxplot(hd95_ed_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
for patch, color in zip(bp3['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[1, 0].set_title('ED HD95 (mm)', fontsize=12)
axes[1, 0].set_ylabel('HD95 (mm)')

# HD95 - ES
hd95_es_data = [
    patient_df['es_hd95_rv'][np.isfinite(patient_df['es_hd95_rv'])],
    patient_df['es_hd95_myo'][np.isfinite(patient_df['es_hd95_myo'])],
    patient_df['es_hd95_lv'][np.isfinite(patient_df['es_hd95_lv'])]
]
bp4 = axes[1, 1].boxplot(hd95_es_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
for patch, color in zip(bp4['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[1, 1].set_title('ES HD95 (mm)', fontsize=12)
axes[1, 1].set_ylabel('HD95 (mm)')

plt.suptitle('nnFormer ACDC Segmentation Metrics Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save the figure
boxplot_path = RESULTS_DIR / "metrics_boxplots.png"
plt.savefig(boxplot_path, dpi=150, bbox_inches='tight')
print(f"✓ Box plots saved to: {boxplot_path}")

plt.show()

## Output Files Summary

The following files are saved in the `results/` directory:

**Predictions** (`results/predictions/`):
- `patient###_frame##.nii.gz` - Segmentation predictions for each test image

**Metrics** (`results/`):
- `metrics_detailed.csv` - Detailed metrics for each image (100 rows)
- `metrics_by_patient.csv` - Patient-level metrics with ED/ES breakdown (50 rows)
- `metrics_summary.csv` - Summary statistics table

**Visualizations** (`results/visualizations/`):
- `patient###_ED_comparison.png` - ED frame comparison visualizations
- `patient###_ES_comparison.png` - ES frame comparison visualizations
- `metrics_boxplots.png` - Box plots of metric distributions

In [ ]:
"""Final summary."""

print("=" * 80)
print("nnFORMER ACDC SEGMENTATION - FINAL SUMMARY")
print("=" * 80)

print(f"\nDataset: ACDC Testing Set")
print(f"Number of patients: {len(patient_df)}")
print(f"Number of images: {len(results_df)}")
print(f"Model: nnFormer (fold_0, model_best)")

print("\n--- Overall Performance ---")
print(f"Mean Dice Score: {patient_df['overall_dice_mean'].mean():.4f} ± {patient_df['overall_dice_mean'].std():.4f}")
mean_hd95, std_hd95 = safe_stats(patient_df['overall_hd95_mean'])
print(f"Mean HD95: {mean_hd95:.2f} ± {std_hd95:.2f} mm")

print(f"\n--- Output Files ---")
print(f"Results directory: {RESULTS_DIR}")
print(f"  - metrics_detailed.csv")
print(f"  - metrics_by_patient.csv")
print(f"  - metrics_summary.csv")
print(f"  - metrics_boxplots.png")
print(f"  - predictions/ ({len(list(PREDICTIONS_DIR.glob('*.nii.gz')))} files)")
print(f"  - visualizations/ ({len(list(VISUALIZATIONS_DIR.glob('*.png')))} files)")

print("\n✓ Processing complete!")

In [ ]:
# Step 5: Export CASC-style debug artifacts
if 'results_df' not in globals() or results_df.empty:
    metrics_path = RESULTS_DIR / "metrics_detailed.csv"
    if metrics_path.exists():
        results_df = pd.read_csv(metrics_path)
    else:
        raise RuntimeError("No metrics available. Run inference and metrics first.")

if 'patient_df' not in globals() or patient_df.empty:
    patient_metrics_path = RESULTS_DIR / "metrics_by_patient.csv"
    if patient_metrics_path.exists():
        patient_df = pd.read_csv(patient_metrics_path)
    else:
        raise RuntimeError("No patient-level metrics available. Build patient summary first.")

debug_report_task = DEBUG_TRACKER.start_task("GENERATE_REPORT", gpu_task=False)

# Build CASC-style metrics snapshot
metrics_snapshot = pd.DataFrame({
    "patient_id": patient_df["patient_id"],
    "model": "nnformer__model",
    "architecture": "nnformer",
    "ed_dice_rv": patient_df.get("ed_dice_rv", np.nan),
    "ed_hd95_rv": patient_df.get("ed_hd95_rv", np.nan),
    "ed_dice_myo": patient_df.get("ed_dice_myo", np.nan),
    "ed_hd95_myo": patient_df.get("ed_hd95_myo", np.nan),
    "ed_dice_lv": patient_df.get("ed_dice_lv", np.nan),
    "ed_hd95_lv": patient_df.get("ed_hd95_lv", np.nan),
    "es_dice_rv": patient_df.get("es_dice_rv", np.nan),
    "es_hd95_rv": patient_df.get("es_hd95_rv", np.nan),
    "es_dice_myo": patient_df.get("es_dice_myo", np.nan),
    "es_hd95_myo": patient_df.get("es_hd95_myo", np.nan),
    "es_dice_lv": patient_df.get("es_dice_lv", np.nan),
    "es_hd95_lv": patient_df.get("es_hd95_lv", np.nan),
    "overall_dice_mean": patient_df.get("overall_dice_mean", np.nan),
})
metrics_snapshot_path = DEBUG_TEXT_DIR / "metrics_snapshot.csv"
metrics_snapshot.to_csv(metrics_snapshot_path, index=False)

# Finalize debug report task and aggregate task profile outputs
DEBUG_TRACKER.end_task(debug_report_task, success=True)
task_df = pd.DataFrame(DEBUG_TRACKER.records)
if task_df.empty:
    task_profile = pd.DataFrame(columns=[
        "process", "tasks", "success_rate", "runtime_mean_s", "runtime_median_s", "runtime_p95_s",
        "cpu_percent_mean", "peak_rss_gb_mean", "peak_rss_gb_max", "runtime_hours_total",
    ])
else:
    task_profile = (
        task_df.groupby("process", as_index=False)
        .apply(
            lambda g: pd.Series(
                {
                    "tasks": int(len(g)),
                    "success_rate": _safe_float(g["success"].mean()),
                    "runtime_mean_s": _safe_float(g["runtime_s"].mean()),
                    "runtime_median_s": _safe_float(g["runtime_s"].median()),
                    "runtime_p95_s": _safe_float(np.percentile(g["runtime_s"], 95)),
                    "cpu_percent_mean": _safe_float(g["cpu_percent"].mean()),
                    "peak_rss_gb_mean": _safe_float(g["peak_rss_gb"].mean()),
                    "peak_rss_gb_max": _safe_float(g["peak_rss_gb"].max()),
                    "runtime_hours_total": _safe_float(g["runtime_s"].sum() / 3600.0),
                }
            )
        )
        .reset_index(drop=True)
        .sort_values("runtime_mean_s", ascending=False)
)

task_profile_path = DEBUG_TEXT_DIR / "task_profile.csv"
task_profile.to_csv(task_profile_path, index=False)

gpu_profile = task_df[task_df.get("gpu_task", False) == True].copy() if not task_df.empty else pd.DataFrame()
if gpu_profile.empty:
    gpu_profile_df = pd.DataFrame(columns=["process", "gpu_tasks", "gpu_walltime_hours", "runtime_median_s"])
else:
    gpu_profile_df = (
        gpu_profile.groupby("process", as_index=False)
        .apply(
            lambda g: pd.Series(
                {
                    "gpu_tasks": int(len(g)),
                    "gpu_walltime_hours": _safe_float(g["runtime_s"].sum() / 3600.0),
                    "runtime_median_s": _safe_float(g["runtime_s"].median()),
                }
            )
        )
        .reset_index(drop=True)
        .sort_values("runtime_median_s", ascending=False)
)

gpu_profile_path = DEBUG_TEXT_DIR / "gpu_profile.csv"
gpu_profile_df.to_csv(gpu_profile_path, index=False)

# Compute debug metrics summary using the same schema as CASC/VSA3L
workflow_duration_seconds = _safe_float(DEBUG_TRACKER.workflow_duration_seconds())
total_tasks = int(len(task_df))
completed_tasks = int(task_df["success"].sum()) if not task_df.empty else 0
failed_tasks = int(total_tasks - completed_tasks)
task_success_rate = _safe_float(completed_tasks / total_tasks) if total_tasks > 0 else float("nan")
retry_rate = 0.0

cpu_time_hours_est = float("nan")
if not task_df.empty:
    valid_cpu = task_df.dropna(subset=["cpu_percent", "runtime_s"]).copy()
    if not valid_cpu.empty:
        cpu_time_hours_est = _safe_float(((valid_cpu["cpu_percent"] / 100.0) * valid_cpu["runtime_s"]).sum() / 3600.0)

memory_gb_hours_est = float("nan")
if not task_df.empty:
    valid_mem = task_df.dropna(subset=["peak_rss_gb", "runtime_s"]).copy()
    if not valid_mem.empty:
        memory_gb_hours_est = _safe_float((valid_mem["peak_rss_gb"] * valid_mem["runtime_s"] / 3600.0).sum())

mean_parallelism_est = float("nan")
if workflow_duration_seconds > 0 and not task_df.empty:
    mean_parallelism_est = _safe_float(task_df["runtime_s"].sum() / workflow_duration_seconds)

n_patients_input = int(len(patient_df["patient_id"].unique()))
n_patients_with_metrics = int(metrics_snapshot["patient_id"].nunique()) if not metrics_snapshot.empty else 0
coverage_rate = _safe_float(n_patients_with_metrics / n_patients_input) if n_patients_input > 0 else float("nan")
patient_throughput_per_hour = _safe_float(n_patients_input * 3600.0 / workflow_duration_seconds) if workflow_duration_seconds > 0 else float("nan")

gpu_task_count = int(len(gpu_profile)) if not gpu_profile.empty else 0
gpu_task_fraction = _safe_float(gpu_task_count / total_tasks) if total_tasks > 0 else float("nan")
gpu_walltime_hours_est = _safe_float(gpu_profile["runtime_s"].sum() / 3600.0) if not gpu_profile.empty else 0.0
avg_gpu_concurrency_est = _safe_float((gpu_walltime_hours_est * 3600.0) / workflow_duration_seconds) if workflow_duration_seconds > 0 else float("nan")

n_models_evaluated = 1
best_model_overall_dice = "nnformer__model"
best_model_dice_value = _safe_float(metrics_snapshot["overall_dice_mean"].mean()) if not metrics_snapshot.empty else float("nan")
cross_model_dice_spread = float("nan")  # Single-model notebook run

overall_rv = pd.concat([metrics_snapshot["ed_dice_rv"], metrics_snapshot["es_dice_rv"]], ignore_index=True) if not metrics_snapshot.empty else pd.Series(dtype=float)
overall_myo = pd.concat([metrics_snapshot["ed_dice_myo"], metrics_snapshot["es_dice_myo"]], ignore_index=True) if not metrics_snapshot.empty else pd.Series(dtype=float)
overall_lv = pd.concat([metrics_snapshot["ed_dice_lv"], metrics_snapshot["es_dice_lv"]], ignore_index=True) if not metrics_snapshot.empty else pd.Series(dtype=float)

rv_dice_mean = _safe_float(overall_rv.mean()) if not overall_rv.empty else float("nan")
myo_dice_mean = _safe_float(overall_myo.mean()) if not overall_myo.empty else float("nan")
lv_dice_mean = _safe_float(overall_lv.mean()) if not overall_lv.empty else float("nan")

architecture_consistency_cv_mean = float("nan")  # Single architecture in this run

summary_payload = {
    "run_name": DEBUG_RUN_NAME,
    "workflow_start": DEBUG_TRACKER.workflow_start_iso,
    "models_requested": "NNFORMER",
    "execution": {
        "workflow_success": failed_tasks == 0,
        "workflow_duration_seconds": workflow_duration_seconds,
        "total_tasks": total_tasks,
        "completed_tasks": completed_tasks,
        "failed_tasks": failed_tasks,
        "task_success_rate": task_success_rate,
        "retry_rate": retry_rate,
        "cpu_time_hours_est": cpu_time_hours_est,
        "memory_gb_hours_est": memory_gb_hours_est,
        "makespan_seconds_trace": workflow_duration_seconds,
        "mean_parallelism_est": mean_parallelism_est,
        "patient_throughput_per_hour": patient_throughput_per_hour,
    },
    "gpu": {
        "gpu_task_count": gpu_task_count,
        "gpu_task_fraction": gpu_task_fraction,
        "gpu_walltime_hours_est": gpu_walltime_hours_est,
        "avg_gpu_concurrency_est": avg_gpu_concurrency_est,
    },
    "scientific": {
        "n_patients_input": n_patients_input,
        "n_patients_with_metrics": n_patients_with_metrics,
        "coverage_rate": coverage_rate,
        "n_models_evaluated": n_models_evaluated,
        "best_model_overall_dice": best_model_overall_dice,
        "best_model_dice_value": best_model_dice_value,
        "cross_model_dice_spread": cross_model_dice_spread,
        "architecture_consistency_cv_mean": architecture_consistency_cv_mean,
        "rv_dice_mean": rv_dice_mean,
        "myo_dice_mean": myo_dice_mean,
        "lv_dice_mean": lv_dice_mean,
    },
}

summary_json_path = DEBUG_TEXT_DIR / "debug_metrics_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2, allow_nan=True)

# Markdown report in the same CASC style
report_lines = [
    "# CASC Debug Report",
    "",
    "## Run Context",
    f"- Run name: {summary_payload['run_name']}",
    f"- Models requested: {summary_payload['models_requested']}",
    f"- Workflow start: {summary_payload['workflow_start']}",
    f"- Workflow success: {summary_payload['execution']['workflow_success']}",
    "",
    "## Execution & Reliability",
    f"- Workflow duration (s): {_safe_float(summary_payload['execution']['workflow_duration_seconds']):.3f}",
    f"- Total tasks: {summary_payload['execution']['total_tasks']}",
    f"- Task success rate: {_safe_float(summary_payload['execution']['task_success_rate']):.4f}",
    f"- Retry rate: {_safe_float(summary_payload['execution']['retry_rate']):.4f}",
    f"- Estimated CPU time (core-hours): {_nan_to_na(summary_payload['execution']['cpu_time_hours_est'], precision=3)}",
    f"- Estimated memory pressure (GB-hours from peak RSS): {_nan_to_na(summary_payload['execution']['memory_gb_hours_est'], precision=3)}",
    f"- Mean parallelism estimate: {_nan_to_na(summary_payload['execution']['mean_parallelism_est'], precision=3)}",
    f"- Patient throughput (patients/hour): {_nan_to_na(summary_payload['execution']['patient_throughput_per_hour'], precision=3)}",
    "",
    "## GPU Scalability",
    f"- GPU task count: {summary_payload['gpu']['gpu_task_count']}",
    f"- GPU task fraction: {_nan_to_na(summary_payload['gpu']['gpu_task_fraction'], precision=4)}",
    f"- Estimated GPU walltime (hours): {_nan_to_na(summary_payload['gpu']['gpu_walltime_hours_est'], precision=3)}",
    f"- Average GPU concurrency estimate: {_nan_to_na(summary_payload['gpu']['avg_gpu_concurrency_est'], precision=3)}",
    "",
    "## Scientific Utility",
    f"- Input patient count: {summary_payload['scientific']['n_patients_input']}",
    f"- Evaluated patient count: {summary_payload['scientific']['n_patients_with_metrics']}",
    f"- Coverage rate: {_nan_to_na(summary_payload['scientific']['coverage_rate'], precision=4)}",
    f"- Models evaluated: {summary_payload['scientific']['n_models_evaluated']}",
    f"- Best model by mean overall Dice: {summary_payload['scientific']['best_model_overall_dice']} ({_nan_to_na(summary_payload['scientific']['best_model_dice_value'], precision=4)})",
    f"- Cross-model Dice spread: {_nan_to_na(summary_payload['scientific']['cross_model_dice_spread'], precision=4)}",
    f"- Architecture consistency CV (lower is better): {_nan_to_na(summary_payload['scientific']['architecture_consistency_cv_mean'], precision=4)}",
    f"- Mean RV Dice: {_nan_to_na(summary_payload['scientific']['rv_dice_mean'], precision=4)}",
    f"- Mean MYO Dice: {_nan_to_na(summary_payload['scientific']['myo_dice_mean'], precision=4)}",
    f"- Mean LV Dice: {_nan_to_na(summary_payload['scientific']['lv_dice_mean'], precision=4)}",
]

report_md_path = DEBUG_TEXT_DIR / "debug_report.md"
report_md_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

# Save simple debug figures to keep same debug folder form factor.
if not task_profile.empty:
    plt.figure(figsize=(14, 6))
    plot_df = task_profile.head(20).copy()
    plt.barh(plot_df["process"], plot_df["runtime_mean_s"], color="#4c78a8")
    plt.gca().invert_yaxis()
    plt.xlabel("Mean Runtime (s)")
    plt.title("Top 20 Processes by Mean Runtime")
    plt.tight_layout()
    plt.savefig(DEBUG_FIGURES_DIR / "process_runtime_top20.png", dpi=150)
    plt.close()

if not metrics_snapshot.empty:
    plt.figure(figsize=(8, 5))
    plt.bar(["nnformer__model"], [metrics_snapshot["overall_dice_mean"].mean()], color="#f58518")
    plt.ylim(0, 1)
    plt.ylabel("Mean Overall Dice")
    plt.title("Mean Overall Dice by Model")
    plt.tight_layout()
    plt.savefig(DEBUG_FIGURES_DIR / "mean_overall_dice_by_model.png", dpi=150)
    plt.close()

print("✓ CASC-style debug artifacts created:")
print(f"  {summary_json_path}")
print(f"  {report_md_path}")
print(f"  {task_profile_path}")
print(f"  {gpu_profile_path}")
print(f"  {metrics_snapshot_path}")
print(f"  {DEBUG_FIGURES_DIR}")